# Global Energy Consumption Analysis

This notebook performs exploratory data analysis on the global energy consumption dataset.

## 1. Import Libraries and Initialize Connectors

In [ ]:
import sys
import pandas as pd
from pathlib import Path

# Add src to path for imports
sys.path.insert(0, str(Path.cwd() / "src"))

from utils import (
    load_kaggle_dataset,
    preprocess_data,
    get_numerical_columns,
    get_categorical_columns,
)
from descriptive_analysis import (
    check_daily_completeness,
    get_numerical_descriptive_stats,
    get_categorical_summary,
    print_descriptive_summary,
    detect_strong_outliers,
    impute_missing_by_month,
)
from visualization import (
    create_time_series_plots,
    create_distribution_plots,
    create_categorical_summary_plots,
    create_summary_statistics_table,
    plot_time_series_decomposition,
    plot_trend_and_components,
)
from modeling import decompose_time_series

c:\Users\gaspa\Desktop\Gaspare\github_portfolio\forecast-global-electricity-price\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load and Prepare Data

In [2]:
# Load dataset from Kaggle
dataset_name = "aramacus/electricity-demand-in-victoria-australia"
df = load_kaggle_dataset(dataset_name)

print(f"Original dataset shape: {df.shape}")
print("\nFirst few rows:")
df.head()

Dataset downloaded to: C:\Users\gaspa\.cache\kagglehub\datasets\aramacus\electricity-demand-in-victoria-australia\versions\2
Original dataset shape: (2106, 14)

First few rows:


,date,demand,RRP,demand_pos_RRP,RRP_positive,demand_neg_RRP,RRP_negative,frac_at_neg_RRP,min_temperature,max_temperature,solar_exposure,rainfall,school_day,holiday
0,2015-01-01,99635.030,25.633696,97319.240,26.415953,2315.790,-7.240000,0.020833,13.3,26.9,23.6,0.0,N,Y
1,2015-01-02,129606.010,33.138988,121082.015,38.837661,8523.995,-47.809777,0.062500,15.4,38.8,26.8,0.0,N,N
2,2015-01-03,142300.540,34.564855,142300.540,34.564855,0.000,0.000000,0.000000,20.0,38.2,26.5,0.0,N,N
3,2015-01-04,104330.715,25.005560,104330.715,25.005560,0.000,0.000000,0.000000,16.3,21.4,25.2,4.2,N,N
4,2015-01-05,118132.200,26.724176,118132.200,26.724176,0.000,0.000000,0.000000,15.0,22.0,30.7,0.0,N,N


In [3]:
# Preprocess the data
df = preprocess_data(df)

print(f"Processed dataset shape: {df.shape}")
print("\nColumn names and types:")
print(df.dtypes)

Processed dataset shape: (2106, 14)

Column names and types:
date               datetime64[us]
demand                    float64
RRP                       float64
demand_pos_RRP            float64
RRP_positive              float64
demand_neg_RRP            float64
RRP_negative              float64
frac_at_neg_RRP           float64
min_temperature           float64
max_temperature           float64
solar_exposure            float64
rainfall                  float64
school_day                    str
holiday                       str
dtype: object


In [4]:
# Get information about columns
numerical_cols = get_numerical_columns(df)
categorical_cols = get_categorical_columns(df)

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")

Numerical columns (11): ['demand', 'RRP', 'demand_pos_RRP', 'RRP_positive', 'demand_neg_RRP', 'RRP_negative', 'frac_at_neg_RRP', 'min_temperature', 'max_temperature', 'solar_exposure', 'rainfall']

Categorical columns (2): ['school_day', 'holiday']


## 3. Perform Descriptive Analysis

In [5]:
# Print comprehensive descriptive summary
print_descriptive_summary(df)

DATASET OVERVIEW
Shape: 2106 rows × 14 columns

MISSING VALUES
        Column  Missing_Count  Missing_Percentage
      rainfall              3                0.14
solar_exposure              1                0.05

NUMERICAL VARIABLES STATISTICS
         Column  Count  Missing          Mean          Std          Min            Q1        Median            Q3           Max   Skewness   Kurtosis
         demand   2106        0 120035.476503 13747.993761 85094.375000 109963.650000 119585.912500 130436.006250 170653.840000   0.186918  -0.334500
            RRP   2106        0     76.079554   130.246805    -6.076028     38.707040     66.596738     95.075012   4549.645105  24.784220 761.384916
 demand_pos_RRP   2106        0 119252.305055 14818.631319 41988.240000 109246.250000 119148.082500 130119.477500 170653.840000  -0.192748   0.666353
   RRP_positive   2106        0     76.553847   130.114184    13.568986     39.117361     66.869058     95.130181   4549.645105  24.850409 764.142399
 dema

In [6]:
# Get detailed statistics for numerical columns\n",
numerical_stats = get_numerical_descriptive_stats(df, numerical_cols)
# Display statistics table visualization
stats_table_fig = create_summary_statistics_table(df, numerical_stats)
stats_table_fig.show()

In [7]:
# Get categorical variables summary
if categorical_cols:
    categorical_summary = get_categorical_summary(df, categorical_cols)

    for col, summary in categorical_summary.items():
        print(f"\n{'=' * 60}")
        print(f"Categorical Variable: {col}")
        print(f"{'=' * 60}")
        print(f"Unique Values: {summary['Unique_Count']}")
        print(f"Missing Values: {summary['Missing']}")
        print("\nValue Counts:")
        value_counts_df = pd.DataFrame(
            [
                {"Value": val, "Count": count}
                for val, count in sorted(
                    summary["Value_Counts"].items(), key=lambda x: x[1], reverse=True
                )
            ]
        )
        print(value_counts_df.to_string(index=False))


Categorical Variable: school_day
Unique Values: 2
Missing Values: 0

Value Counts:
Value  Count
    Y   1453
    N    653

Categorical Variable: holiday
Unique Values: 2
Missing Values: 0

Value Counts:
Value  Count
    N   2029
    Y     77


In [8]:
# Determine if there's a time column
time_cols = [
    col for col in df.columns if "year" in col.lower() or "date" in col.lower()
]
time_col = time_cols[0] if time_cols else None
check_date_col = check_daily_completeness(df, time_col, verbose=True) 

---- Daily Completeness Check ----
Start date: 2015-01-01 00:00:00
End date:   2020-10-06 00:00:00
Expected days: 2106
Actual days:   2106
Missing days:  0
Duplicate days:0
Complete:      True


## 4. Generate Visualizations

In [9]:
# Create time series plots for numerical columns

time_series_fig = create_time_series_plots(
    df, time_col=time_col, numerical_cols=numerical_cols
)
time_series_fig.show()

In [10]:
# Create distribution plots (histograms) for numerical columns
distribution_fig = create_distribution_plots(df, numerical_cols=numerical_cols)
distribution_fig.show()

In [11]:
# Create categorical variable visualizations
if categorical_cols:
    categorical_figs = create_categorical_summary_plots(df, categorical_cols)

    for col, fig in categorical_figs.items():
        print(f"\nDisplaying categorical visualization for: {col}")
        fig.show()


Displaying categorical visualization for: school_day



Displaying categorical visualization for: holiday


## Outlier Detection

In [12]:
# Detect strong outliers
outliers_df = detect_strong_outliers(df, columns=["RRP"], z_score_threshold=4)


STRONG OUTLIERS DETECTED (z-score > 4)
Found 8 rows with outliers

           date      demand          RRP  demand_pos_RRP  RRP_positive  demand_neg_RRP  RRP_negative  frac_at_neg_RRP  min_temperature  max_temperature  solar_exposure  rainfall school_day holiday
1113 2018-01-18  154648.065  1210.137920      154648.065   1210.137920             0.0           0.0              0.0             16.6             40.0            30.8       0.0          N       N
1114 2018-01-19  165070.595   647.574163      165070.595    647.574163             0.0           0.0              0.0             22.5             40.3            30.6       0.0          N       N
1133 2018-02-07  159307.315   624.260934      159307.315    624.260934             0.0           0.0              0.0             18.7             37.4            27.1       0.0          Y       N
1484 2019-01-24  155891.345  4549.645105      155891.345   4549.645105             0.0           0.0              0.0             17.3          

## Impute Missing Values by Month

In [13]:
# Impute missing values by month
df_imputed = impute_missing_by_month(df)


IMPUTING MISSING VALUES BY MONTH

Column: rainfall
Missing values: 3
  Month 6: imputed 1 values (avg: 1.23)
  Month 10: imputed 2 values (avg: 1.09)

Column: solar_exposure
Missing values: 1
  Month 11: imputed 1 values (avg: 20.65)



## Summary

This analysis provided a comprehensive overview of the global energy consumption dataset including:
- **Missing Values Analysis**: Identification of missing data patterns
- **Descriptive Statistics**: Mean, median, standard deviation, and distribution shape for all numerical variables
- **Time Series Analysis**: Visualization of numerical variables over time
- **Distribution Analysis**: Histogram plots showing the distribution of each numerical variable
- **Categorical Analysis**: Frequency distribution and unique values for categorical variables
- **Missings imputed**: Imputation of missing values based on climate averages of the same month over time - missings were in rainfall and solar_exposure
- **Outliers**: A few strong outliers have been detected in RRP. They are happening mostly during heat waves. I will keep an eye on that.

## Time Series Decomposition Analysis

### RRP Decomposition

In [14]:
# Decompose RRP
rrp_decomposition = decompose_time_series(
    df_imputed, "RRP", date_col=time_col, seasonal_periods=(7, 365)
)

In [15]:
# Plot RRP decomposition
rrp_decomp_fig = plot_time_series_decomposition(rrp_decomposition, "RRP")
rrp_decomp_fig.show()

### Demand Decomposition

In [16]:
# Decompose Demand
demand_decomposition = decompose_time_series(
    df_imputed, "demand", date_col=time_col, seasonal_periods=(7, 365)
)

In [17]:
# Plot Demand decomposition
demand_decomp_fig = plot_time_series_decomposition(demand_decomposition, "Demand")
demand_decomp_fig.show()

In [18]:
# Plot original series with trend overlay
rrp_trend_fig = plot_trend_and_components(rrp_decomposition, "RRP")
demand_trend_fig = plot_trend_and_components(demand_decomposition, "Demand")

rrp_trend_fig.show()
demand_trend_fig.show()